This code loads the MultiEURLEX dataset. Retrieves the English descriptors for all EuroVoc concept labels from the official repository and maps them to their corresponding label IDs. 

In [1]:
# Load full dataset with all 7,390 labels
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import requests

print("Loading dataset with ALL labels (7,390)...")
dataset = load_dataset('coastalcph/multi_eurlex', 'en', split='test', 
                       label_level='all_levels', trust_remote_code=True)

# Get label descriptors
print("Loading EuroVoc descriptors...")
url = "https://raw.githubusercontent.com/nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
eurovoc_concepts = requests.get(url).json()

# Get all label IDs and descriptors
classlabel = dataset.features["labels"].feature
label_ids = classlabel.names
label_descriptors = [eurovoc_concepts[label_id]['en'] for label_id in label_ids]

print(f"✓ Dataset: {len(dataset)} documents")
print(f"✓ Labels: {len(label_descriptors)}")
print(f"\nSample labels:")
for i in [0, 100, 500, 1000, 5000]:
    print(f"  {i}: {label_ids[i]} → {label_descriptors[i]}")

Loading dataset with ALL labels (7,390)...
Loading EuroVoc descriptors...
✓ Dataset: 5000 documents
✓ Labels: 7390

Sample labels:
  0: 100149 → social questions
  100: 100182 → organisation of the legal system
  500: 2190 → Latin American organisation
  1000: 1162 → alcoholism
  5000: 1476 → interest


Loads multilingual sentence embedding model (E5-small) and generates dense vector representations for all 7390 EuroVoc label desciptors. 

In [2]:
# Load model and embed all 7,390 labels
print("\nLoading embedding model...")
model = SentenceTransformer('intfloat/multilingual-e5-small')

print("\nEmbedding all 7,390 labels...")
label_embeddings = model.encode(label_descriptors, show_progress_bar=True, batch_size=32)
print(f"✓ Label embeddings shape: {label_embeddings.shape}")


Loading embedding model...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ad4948fe-9803-4232-a805-0b79e4a6fc43)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].



Embedding all 7,390 labels...


Batches:   0%|          | 0/231 [00:00<?, ?it/s]

✓ Label embeddings shape: (7390, 384)


Evaluates a zero-shot label retrieval system by testing it on the first docuemnt on the dataset. Embeds the document text using same model (E5-small), computes cosine similarities between document embedding and all 7390 label embeddings, and retrieves the top-10 most similar labels. Then compares retrieved labels against the true labels to calculate Precision@10, which measure how many of the predicted labels are actually correct. 

In [4]:
# Test on first document
doc_idx = 0
doc = dataset[doc_idx]

print("="*80)
print(f"DOCUMENT {doc_idx}")
print("="*80)
print(f"Text: {doc['text'][:300]}...\n")

# True labels
true_labels = doc['labels']
true_label_names = [label_descriptors[i] for i in true_labels]
print(f"True labels ({len(true_labels)}):")
for i, name in enumerate(true_label_names, 1): 
    print(f"  {i}. {name}")

# Embed and retrieve
doc_embedding = model.encode(doc['text'])
similarities = cosine_similarity([doc_embedding], label_embeddings)[0]

k = 10
top_k_indices = np.argsort(similarities)[::-1][:k]

print(f"\n{'='*80}")
print(f"TOP {k} RETRIEVED LABELS")
print(f"{'='*80}")
for rank, idx in enumerate(top_k_indices, 1):
    is_correct = "✓" if idx in true_labels else " "
    print(f"{rank:2d}. {is_correct} {label_descriptors[idx]:50s} {similarities[idx]:.4f}")

# Evaluate
correct = len(set(top_k_indices) & set(true_labels))
precision = correct / k
print(f"\nPrecision@{k}: {precision:.4f} ({100*precision:.1f}%)")

DOCUMENT 0
Text: COUNCIL REGULATION (EU) No 1390/2013
of 16 December 2013
on the allocation of fishing opportunities under the Protocol agreed between the European Union and the Union of the Comoros setting out the fishing opportunities and financial contribution provided for in the Fisheries Partnership Agreement c...

True labels (10):
  1. France
  2. agreement (EU)
  3. fishing permit
  4. fishing agreement
  5. Portugal
  6. protocol to an agreement
  7. Comoros
  8. fishing area
  9. fishing rights
  10. Spain

TOP 10 RETRIEVED LABELS
 1. ✓ fishing agreement                                  0.8706
 2.   fishing regulations                                0.8692
 3.   common fisheries policy                            0.8606
 4.   European Fisheries Control Agency                  0.8569
 5.   European fisheries fund                            0.8566
 6.   fisheries policy                                   0.8550
 7. ✓ fishing rights                                     0.8494
 8.  